In [1]:
from pathlib import Path
import shutil
import numpy as np
from tqdm import tqdm
import pandas as pd

import json

import xml.etree.ElementTree as ET
import re
from functools import wraps
from typing import Callable, Union, Any

from IPython.display import clear_output


In [3]:
import jpype
import scyjava

try:
    scyjava.config.endpoints.append('ome:formats-gpl:latest')
    scyjava.start_jvm()
    loci = jpype.JPackage("loci")
    loci.common.DebugTools.setRootLevel("WARN")
    print("Bio-Formats logging level set to WARN.")

except ImportError:
    print("Could not import jpype or scyjava. Make sure they are installed.")
except jpype.JException as e:
    print(f"An error occurred while trying to configure Java logging: {e}")

from bioio import BioImage
import bioio_bioformats


Bio-Formats logging level set to WARN.


In [ ]:
if jpype.isJVMStarted():
    jpype.shutdownJVM()

In [4]:

def handle_return_decorator(func: Callable) -> Callable:
    """
    Decorator to handle return values from XML metadata search.
    Standardizes the output format for dictionary and string returns.
    """
    @wraps(func)
    def wrapper(*args, **kwargs) -> Union[dict, str, None]:
        result = func(*args, **kwargs)
        if isinstance(result, dict):
            if len(result) == 1:
                return list(result.values())[0]
            else:
                return result
        if isinstance(result, str):
            return result
        return None
    return wrapper

def clean_up_qname(input_str: str) -> str:
    """Remove XML namespace from qname."""
    return re.sub(r'{[^}]*}*', '', input_str)

@handle_return_decorator
def find_metadata_in_xml(image: Any, text_to_search: str) -> Union[str, dict, None]:
    """
    Searches for the first occurrence of text_to_search in qnames or fnames.

    Args:
        image: The BioImage object containing OME metadata.
        text_to_search: The text to search for in qnames and fnames.

    Returns:
        Union[str, dict, None]: The value as a string or dict if found, otherwise None.
    """
    try:
        xml_list = [xml_annotation for xml_annotation in image.ome_metadata.structured_annotations]

        # Search in qnames (first annotation)
        if xml_list:
            for elem in xml_list[0].value.any_elements:
                # Handle elements with children
                if getattr(elem, "children", None):
                    for child in elem.children:
                        if text_to_search in clean_up_qname(getattr(child, "qname", "")):
                            return child.attributes
                # Handle elements without children
                elif text_to_search in clean_up_qname(getattr(elem, "qname", "")):
                    return elem.attributes

        # Search in fnames (second annotation)
        if len(xml_list) > 1:
            for elem in xml_list[1].value.any_elements:
                if getattr(elem, "children", None):
                    for child in elem.children:
                        if text_to_search in child.attributes.get('fname', ''):
                            return child.attributes.get('Value', '')

    except Exception as e:
        print(f"Error during XML annotation search: {e}")

    return None


In [ ]:
image = BioImage("output/AG-29_Anika/221031_AG-029_A1_zoom4-1_z5_16-21-00.ome.tif", reader=bioio_bioformats.Reader)

xml_list = [xml_annotation for xml_annotation in image.ome_metadata.structured_annotations]

with open("first.xml", "w") as f:
    f.write(xml_list[0].to_xml())

with open("second.xml", "w") as f:
    f.write(xml_list[1].to_xml())

result = find_metadata_in_xml(image, "Filter0")
print(f'{result} ({type(result)})')

result = find_metadata_in_xml(image, "FilterAxis")
print(f'{result} ({type(result)})')

result = find_metadata_in_xml(image, "DataAxis3")
print(f'{result} ({type(result)})')

result = find_metadata_in_xml(image, "Blaze NA")
print(f'{result} ({type(result)})')

result = find_metadata_in_xml(image, "InstrumentMode")
print(f'{result} ({type(result)})')

result = find_metadata_in_xml(image, "blah")
print(f'{result} ({type(result)})')

result = find_metadata_in_xml(image, "Blaze ObjectiveMagnification")
print(f'{result} ({type(result)})')

In [5]:
def extract_metadata(image_path: Path) -> dict:
    
    image = BioImage(image_path, reader=bioio_bioformats.Reader)
    
    try: 
        metadata_dict = {
            'Obj Magnfication': "Blaze ObjectiveMagnification",
            'Obj NA': "ObjectiveNA",
            'Digital Zoom': "Blaze CurrentZoom",
            'Measurament TimeStamp': "MeasTime",    
        }    

        for key in metadata_dict.keys():
            if isinstance(metadata_dict[key], str):
                metadata_dict[key] = find_metadata_in_xml(image, metadata_dict[key])

        for i in range(len(image.channel_names)):
            tmp_filter_dict = find_metadata_in_xml(image, f'Filter{i}')
            metadata_dict[f'Channel{i}_EXC [nm]'] = tmp_filter_dict['ExcitationWL']
            metadata_dict[f'Channel{i}_EM [nm]'] = tmp_filter_dict['EmissionWL']

        metadata_dict = {
            'File Name': image_path.name,
            'File Path': str(image_path.parent.resolve()),
            'X_Y_Z [pixels]': f'{image.dims.X} x {image.dims.Y} x {image.dims.Z}',
            'Z_step [um]': image.physical_pixel_sizes.Z,
            'Number of Channels': len(image.channel_names),
            **metadata_dict,
        }
        
        
    except Exception as e:
        metadata_dict = None
        print(f"❌ Failed to process image {image_path.name}: {e}")
    
    return metadata_dict




In [6]:
manual_metadata = {
    # Should extract from Excel file
    'Exp ID': 'AG-029',
    'Exp Title': 'Lightsheet Tibia',
    'Project Date': '22.11.2022',
    'Project Technology': 'LSFM',
    'Project Title': 'Cooperation Iannis Adamopoulos',
    'Other Comments': 'CD61-PE, CD31-AF647, Ly6G APC-eFluor780',
    
    # Additional Metadata
    'Host': 'Mouse',
    'Location': 'Tibia',
}

In [7]:
files = [p for p in sorted(Path('output/AG-29_Anika/NEW').glob('*.ome.tif'))]

complete_list_metadata = []

for image_file in tqdm(files, desc="Processing samples", unit="sample"):
    tmp = extract_metadata(image_file)
    
    if tmp:
        tmp = {**manual_metadata, **tmp}
        complete_list_metadata.append(tmp)
        
df = pd.DataFrame.from_dict(complete_list_metadata)
#df.to_csv('new_anika.csv', index=False)
    

Processing samples: 100%|██████████| 1/1 [00:05<00:00,  5.32s/sample]


In [8]:
df

,Exp ID,Exp Title,Project Date,Project Technology,Project Title,Other Comments,Host,Location,File Name,File Path,...,Digital Zoom,Measurament TimeStamp,Channel0_EXC [nm],Channel0_EM [nm],Channel1_EXC [nm],Channel1_EM [nm],Channel2_EXC [nm],Channel2_EM [nm],Channel3_EXC [nm],Channel3_EM [nm]
0,AG-029,Lightsheet Tibia,22.11.2022,LSFM,Cooperation Iannis Adamopoulos,"CD61-PE, CD31-AF647, Ly6G APC-eFluor780",Mouse,Tibia,221031_AG-029_A1_zoom12-1_z5_16-53-36.ome.tif,/mnt/eternus/users/Davide/blaze-2-ome/output/A...,...,1.000000,2022-10-31T16:53:38,470,525,560,650,630,680,630,845


In [13]:
# Employ to OME-ZARR for AGAVE (csv)

#df = pd.read_csv('anika.csv', index_col=False)

df = pd.DataFrame.from_dict(complete_list_metadata)

linux_path = str(Path("/mnt/eternus/users/Davide/blaze-2-ome/output/AG-29_Anika"))
windows_path = str(Path(r"\\ambiom-fs1.isas.de\ambiom_storage\users\Davide\blaze-2-ome\output\AG-29_Anika\ZARR"))

df['File Path'] = df['File Path'].str.replace(linux_path, windows_path, regex=False)
df['File Name'] = df['File Name'].str.replace('ome.tif', 'ome.zarr', regex=False)
df['File Path'] = df['File Path'] + "\\" +df['File Name']

df.to_csv('AG29_windows_zarr.csv', index=False)